# 📊 FAI Dimension & Methodology Validation

This notebook performs methodological validation on the **Financial Access Index (FAI)** calculation pipeline across 1,000 synthetic financial profiles.

### Key Objectives:
1. **Dimension Score Calculation**: Run `FAIDimensionPipeline` across 1,000 synthetic profiles.
2. **Distribution & Radar Analysis**: Visualize score distributions and radar charts across 4 socio-economic archetypes.
3. **Correlation Heatmaps**: Evaluate Pearson and Spearman rank correlations among the 7 FAI dimensions.
4. **Sensitivity Analysis**: Demonstrate mathematical continuity without abrupt score jumps in `[0.00, 100.00]`.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
sys.path.insert(0, os.path.abspath(".."))

from src.domain.access_index.dimension_calculators import FAIDimensionPipeline
from src.domain.access_index.weighting import DeterministicLinearWeightingStrategy
from src.domain.access_index.dimensions import WeightVector
from src.domain.shared.value_objects import DimensionKey

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.size'] = 11


## 1. Load Synthetic Profiles Dataset

In [ ]:
dataset_path = os.path.join("..", "data", "synthetic", "profiles_dataset.csv")
if not os.path.exists(dataset_path):
    from data.synthetic.profile_generator import generate_synthetic_profiles
    df_raw = generate_synthetic_profiles(1000)
else:
    df_raw = pd.read_csv(dataset_path)

print(f"Loaded {len(df_raw)} financial profiles.")
df_raw.head()


## 2. Compute FAI Dimension & Composite Scores

In [ ]:
pipeline = FAIDimensionPipeline()
weighting_strategy = DeterministicLinearWeightingStrategy()
weights = WeightVector.default_equal_weights()

dim_scores_list = []
composite_scores = []

for _, row in df_raw.iterrows():
    feature_dict = row.to_dict()
    dims = pipeline.compute_all_dimensions(feature_dict)
    comp_score = weighting_strategy.compute_composite_score(dims, weights)
    
    score_entry = {k.value: float(v.score.value) for kk, (k, v) in enumerate(dims.items())}
    score_entry["overall_fai_score"] = float(comp_score.value)
    score_entry["archetype"] = row["archetype"]
    dim_scores_list.append(score_entry)

df_scores = pd.DataFrame(dim_scores_list)
print("Calculated score breakdown summary:")
df_scores.groupby("archetype")["overall_fai_score"].describe()


## 3. Score Distributions Across 7 Dimensions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

dim_columns = [k.value for k in DimensionKey] + ["overall_fai_score"]

for i, col in enumerate(dim_columns):
    sns.histplot(df_scores[col], kde=True, ax=axes[i], color="indigo", bins=20)
    axes[i].set_title(f"Distribution: {col.replace('_', ' ').title()}", fontsize=12, fontweight="bold")
    axes[i].set_xlim(0, 100)

plt.tight_layout()
plt.show()


## 4. Radar / Spider Chart by Archetype

In [ ]:
archetype_means = df_scores.groupby("archetype")[[k.value for k in DimensionKey]].mean()

categories = [k.value.capitalize() for k in DimensionKey]
N = len(categories)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
plt.xticks(angles[:-1], categories, color='grey', size=11)

colors = {"ARCHETYPE_A_HIGH_ACCESS": "#2ecc71", "ARCHETYPE_B_FEE_BURDENED": "#e67e22",
          "ARCHETYPE_C_LOW_RESILIENCE": "#e74c3c", "ARCHETYPE_D_DISCONNECTED": "#95a5a6"}

for arch in archetype_means.index:
    values = archetype_means.loc[arch].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=arch, color=colors.get(arch, "blue"))
    ax.fill(angles, values, color=colors.get(arch, "blue"), alpha=0.1)

plt.title("FAI 7 Dimensions Profile by Socio-Economic Archetype", size=14, fontweight="bold", y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.show()


## 5. Pearson & Spearman Correlation Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pearson_corr = df_scores[[k.value for k in DimensionKey]].corr(method="pearson")
spearman_corr = df_scores[[k.value for k in DimensionKey]].corr(method="spearman")

sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="Blues", ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title("Pearson Linear Correlation Matrix", fontweight="bold")

sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="Purples", ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title("Spearman Rank Correlation Matrix", fontweight="bold")

plt.tight_layout()
plt.show()


## 6. Sensitivity Analysis (Mathematical Continuity Test)

In [ ]:
base_features = {
    "wallet_count": 2, "ilp_reachable": True, "tx_success_rate": 0.85,
    "avg_connection_latency_ms": 250.0, "fee_to_volume_ratio": 0.02,
    "settlement_fulfillment_rate": 0.90, "cross_asset_success_rate": 0.80,
    "tx_frequency_monthly": 10, "tx_volume_monthly_usd": 150.0,
    "reserve_liquidity_usd": 40.0, "fallback_route_available": True
}

latencies = np.linspace(50, 2000, 100)
scores_sensitivity = []

for lat in latencies:
    f = base_features.copy()
    f["avg_connection_latency_ms"] = float(lat)
    dims = pipeline.compute_all_dimensions(f)
    comp = weighting_strategy.compute_composite_score(dims, weights)
    scores_sensitivity.append(float(comp.value))

plt.figure(figsize=(10, 5))
plt.plot(latencies, scores_sensitivity, color="darkred", linewidth=2.5)
plt.title("Sensitivity Curve: Latency (ms) vs Overall FAI Score", fontweight="bold")
plt.xlabel("Average Connection Latency (ms)")
plt.ylabel("Overall FAI Score [0.00, 100.00]")
plt.ylim(0, 100)
plt.axvline(200, color="gray", linestyle="--", label="Penalty Trigger (200ms)")
plt.legend()
plt.show()
